# Chapter 6 &mdash; The Language Stethoscope: Indistinguishability

**Concept 6 of the Chapter 6 decomposition:** *The Language Stethoscope: Indistinguishability and Minimality*

The language of a <i>state</i> is what it accepts as a start state; states with the same one can merge.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter6/Concept-Language-Stethoscope/Concept-Language-Stethoscope.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *

import jove; print('Jove loaded from', list(jove.__path__)[0])

## 1. The idea


Put a stethoscope on a single state $q$: **the language of $q$** is what the machine
would accept if $q$ were the start state.

Two states are **indistinguishable** if their languages are equal &mdash; no string can
tell them apart from where they stand. Such states can be **merged** without changing
the machine's language, and a DFA is **minimal** exactly when no two states are
indistinguishable.

The finite test: instead of comparing infinite languages, compare behaviour on all
strings up to length $|Q|-1$, which turns out to be enough (Concept 8).

## 2. Definitions

### A machine with redundancy built in

In [ ]:
D = md2mc('''DFA
IF : 0 -> A
IF : 1 -> B
A  : 0 -> IF
A  : 1 -> C
B  : 0 -> C
B  : 1 -> IF
C  : 0 -> B
C  : 1 -> A
''')

### The language of a state, up to a length bound

In [ ]:
from itertools import product
def lang_of_state(D, q, n):
    out = set()
    for k in range(n+1):
        for p in product(sorted(D["Sigma"]), repeat=k):
            s = ''.join(p)
            if run_dfa_h(D, s, q) in D["F"]:
                out.add(s)
    return out

### Indistinguishability, decided by comparing those languages

In [ ]:
def indist_pairs(D, n=None):
    n = n if n is not None else len(D["Q"])
    L = {q: lang_of_state(D, q, n) for q in D["Q"]}
    qs = sorted(D["Q"])
    return [(a, b) for i, a in enumerate(qs) for b in qs[i+1:] if L[a] == L[b]]

<!-- nav-strip -->

---

&larr;&nbsp;[Ch6&nbsp;5.&nbsp;Isomorphism = Language Equivalence + Equal State Count](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter6/Concept-Isomorphism-Vs-Equivalence/Concept-Isomorphism-Vs-Equivalence.ipynb) &nbsp;&middot;&nbsp; [**Chapter 6** index](https://github.com/ganeshutah/Jove/blob/master/Chapter6/README.md) &nbsp;&middot;&nbsp; [Ch6&nbsp;7.&nbsp;The Myhill–Nerode Theorem, and the Formal Definition of DFA Isomorphism](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter6/Concept-Myhill-Nerode/Concept-Myhill-Nerode.ipynb)&nbsp;&rarr;

---

## 3. Tests

Each state's language, listed short.

In [ ]:
for q in sorted(D["Q"]):
    L = sorted(lang_of_state(D, q, 3), key=lambda s: (len(s), s))
    print("%-4s accepts (len<=3): %s" % (q, L[:8]))

Indistinguishable pairs, found by comparing those languages.

In [ ]:
pairs = indist_pairs(D)
print("indistinguishable pairs :", pairs)
print("so the minimal machine should have %d states; min_dfa says %d"
      % (len(D["Q"]) - len(pairs), len(min_dfa(D)["Q"])))

Merging an indistinguishable pair preserves the language &mdash; `min_dfa` does exactly that.

In [ ]:
m = min_dfa(D)
print("original %d states -> minimal %d states" % (len(D["Q"]), len(m["Q"])))
assert langeq_dfa(D, m)
print("same language after merging? ", langeq_dfa(D, m))

A **minimal** machine has no indistinguishable pairs &mdash; that is the definition.

In [ ]:
print("indistinguishable pairs in the minimal machine :", indist_pairs(m))
assert indist_pairs(m) == []

Distinguishing a pair means exhibiting a string; here is one.

In [ ]:
qs = sorted(m["Q"])
if len(qs) >= 2:
    a, b = qs[0], qs[1]
    La, Lb = lang_of_state(m, a, len(m["Q"])), lang_of_state(m, b, len(m["Q"]))
    w = sorted(La ^ Lb, key=lambda s: (len(s), s))[0]
    print("%s and %s are separated by %r : %s vs %s"
          % (a, b, w, run_dfa_h(m, w, a) in m["F"], run_dfa_h(m, w, b) in m["F"]))

## 4. Exercises


1. What is the language of a black-hole state? Of a state all of whose successors are final?
2. Show that indistinguishability is an equivalence relation.
3. Why is length $|Q|-1$ enough? (Concept 8 answers this.)

In [ ]:
# Your work for the exercises above.

## 5. Where next

In [ ]:
# Previous / next, and a search box for all 245 concepts.
# Type a chapter (Chapter7, ch7) or words from a title (pumping, subset).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:  load_here('Chapter7/Concept-...')
from jove.Nav import nav, load_here
nav(here='Chapter6/Concept-Language-Stethoscope')